# GEMS E6 — deadline-first full submission workflow (cached, no tests, no mirrors)

**Objective:** generate a trustworthy submission artifact today, using existing model predictions and without re-running the 1 m/2 m DEM pipeline. Reuse the previously validated W2 champion as the **primary** candidate. E6 is provided as a separate masked-gating model module, but **must not replace the champion** until its checkpoints and DTI validation exist.

This notebook is self-contained on Colab: mount Drive, load branch code, verify the existing W2 GeoTIFF against frozen checksum and previously passed hard checks, persist status, and optionally download the GeoTIFF for manual DrivenData upload. No benchmark/test suite, no new DEM extraction, no duplicated data mirror, no automatic upload.

**Known E1 state (last inspected):** CV 0 and CV 1 checkpoints exist, CV 2 is missing; E1 train stage remains unfinished. This notebook will report that blocker rather than claim the E6 network is ready. Progress and cache stay in the original Drive directories.


In [ ]:
# =========================== CONFIG ============================
REPO = "mybenkhadda/GEMS---drivendata"
REF = "e6-submission-fasttrack"
DRIVE_ROOT = "/content/drive/MyDrive/GEMS_2"
LOCAL_REPO = "/content/GEMS_E6"
OUT_REL = "outputs/e6_submission_milestone"
# Full submission *artifact* pipeline is enabled by default.
PREPARE_SUBMISSION_TODAY = True
CHECK_E6_READINESS = True
# Disabled by design: unfinished E1 baseline, no complete E6 trained checkpoint.
TRAIN_E6 = False
AUTO_UPLOAD = False     # always manual; do not insert credentials
RUN_TESTS = False       # user-requested: no synthetic test suite or benchmarking loops
CREATE_MIRRORS = False # user-requested: never copy/duplicate giant GeoTIFF data
DOWNLOAD_TIF_TO_BROWSER = True


## 01 — Bootstrap a single versioned repo in Colab

Uses the chosen branch, not the previous G07 main branch. The repo checkout is source code only, not a data mirror. Keep the DEM and champion GeoTIFFs in the original `GEMS_2` directories.


In [ ]:
from pathlib import Path
import subprocess, sys, json
from google.colab import drive
from IPython.display import display, JSON

drive.mount("/content/drive")
assert Path(DRIVE_ROOT).is_dir(), f"Mount Drive correctly: {DRIVE_ROOT}"
repo = Path(LOCAL_REPO)
if not (repo / ".git").is_dir():
    subprocess.run(["git","clone","--depth","1","--branch",REF,
                    f"https://github.com/{REPO}.git",str(repo)],check=True)
else:
    subprocess.run(["git","-C",str(repo),"fetch","--depth","1","origin",REF],check=True)
    subprocess.run(["git","-C",str(repo),"checkout","--detach","FETCH_HEAD"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(repo)+"[colab]"],check=True)
if str(repo) not in sys.path:
    sys.path.insert(0,str(repo))
HEAD = subprocess.check_output(["git","-C",str(repo),"rev-parse","HEAD"],text=True).strip()
print("SOURCE:",HEAD)
print("Tests:",RUN_TESTS,"Mirrors:",CREATE_MIRRORS,"Auto-upload:",AUTO_UPLOAD)


## 02 — Recover the frozen W2 champion and write a final readiness manifest

No retraining and no duplicate GeoTIFF. A **frozen SHA256** must match the independently verified W2 champion from the previous GEMS submission factory; all previously recorded hard validations must have passed. The code also rereads the raster in blocks to check its resolution, CRS, dtype, value range and non-empty predictions. If anything differs, it blocks the upload.


In [ ]:
from gems_bench.e6.submission_fasttrack import prepare_verified_fallback
if not PREPARE_SUBMISSION_TODAY:
    raise RuntimeError("Set PREPARE_SUBMISSION_TODAY=True to produce today's validated submission manifest")
SUB = prepare_verified_fallback(DRIVE_ROOT, output_folder=OUT_REL)
print("STATUS:",SUB["status"])
print("SUBMISSION TIFF:",SUB["submission_file"])
print("SHA256:",SUB["original_sha256"])
print("Frozen hard checks passed:",SUB["previous_hard_checks_passed"])
print("Positive prediction pixels:",SUB["new_raster_readback"]["positive_px"])
print("Artifact bytes remain in the ORIGINAL Drive folder (no mirror).")
display(JSON(SUB))


## 03 — E6 research readiness (informational only)

Reads existing E1 fold checkpoints and G07 cached 100 m DEM manifests; saves one JSON status file. It cannot promote untrained E6 predictions. See `gems_bench/models/geostructurenet_e6.py` for the new warm-started two-way-to-three-way model with guaranteed zero-DEM pixel fallback.


In [ ]:
from gems_bench.e6.readiness import report_e6_readiness
if CHECK_E6_READINESS:
    E6 = report_e6_readiness(DRIVE_ROOT, output_folder=OUT_REL)
    print("Warm-start baseline folds:", E6["available_cv_checkpoints"])
    print("Missing baseline folds:", E6["missing_cv_checkpoints"])
    print("E6 ready for full CV:", E6["ready_to_train_full_e6_cv"])
    print("E6 submission eligible:", E6["ready_for_e6_submission"])
    display(JSON(E6))
if TRAIN_E6:
    raise RuntimeError("E6 is NOT submission-ready. Finish the missing E1 fold and create fold-isolated DEM datasets before enabling training. Baseline W2 submission remains ready.")


## 04 — Download the validated submission for manual upload

This downloads the **original** GeoTIFF directly from Drive; it does not create a second copy on Drive. Upload the TIFF (not the JSON, CSV, or GitHub notebook) via the DrivenData competition submission page. Manual upload is required; this notebook never uploads automatically.


In [ ]:
from pathlib import Path
if SUB["status"] != "READY_FOR_MANUAL_UPLOAD":
    raise RuntimeError("Submission is not ready; do not upload")
print("Ready file:",SUB["submission_file"])
print("Manifest:",Path(DRIVE_ROOT)/OUT_REL/"ready_for_manual_upload.json")
if DOWNLOAD_TIF_TO_BROWSER:
    from google.colab import files
    files.download(SUB["submission_file"])


## 05 — Success conditions and next actions

- Submission file must pass all hard validation and frozen SHA256 checks.
- Mark it ready for **manual upload**, not uploaded.
- If E1 fold 2 is still missing, E6 three-way training must remain blocked, not silently substituted.
- After manual upload, record the public leaderboard score in your experiment log. Do not claim an E6 result until E6 itself is trained and independently validated.
- Outputs: only small `outputs/e6_submission_milestone/ready_for_manual_upload.json`, `submission_inventory.csv`, and `e6_readiness.json`; source TIFF and checkpoints are referenced in place.
